[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flexfengfeng/dsai-m3-ml-genai/blob/main/L10-transformers-genai/notebooks/03_using_an_llm.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. The repo's `.env` file handles thread caps automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [1]:
# === Colab-compat setup (no-op when running locally) ===
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
for _key, _val in [("OMP_NUM_THREADS", "1"), ("MKL_NUM_THREADS", "1"), ("TOKENIZERS_PARALLELISM", "false")]:
    os.environ.setdefault(_key, _val)


# L10 · NB 03 — Using a small LLM

> *We have the architecture. Now let's load a generative transformer and have it produce text. Same Lego bricks as NB 02 — but trained to predict the next token, with a causal mask and an instruction-tuning phase.*

Today's model: `SmolLM2-360M-Instruct` from Hugging Face. 361M parameters, ~720 MB on disk, runs on CPU in 1-3 seconds per response.

In this notebook we will:

1. Load the model and its tokenizer
2. Watch tokenisation happen end-to-end
3. Generate text — first with greedy decoding, then with sampling
4. Tweak temperature, top-k, top-p and see how they change output
5. Use chat templates the right way for instruction-tuned models

## 1 · Setup

In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'

import warnings
warnings.filterwarnings('ignore')

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(1)
torch.manual_seed(0)

MODEL_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

Loading HuggingFaceTB/SmolLM2-360M-Instruct...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded in 2.8s
Parameters: 361,821,120
Vocab size: 49,152


## 2 · Tokenisation, step by step

Before the model sees any text, the tokenizer chops it into subword IDs. Let's see exactly what happens for one prompt.

In [3]:
text = "What's a good summer dress under £80?"

# Step 1: text -> token strings (for human readability)
token_strs = tokenizer.tokenize(text)
print(f"Input text: {text!r}")
print(f"Tokens    : {token_strs}")

# Step 2: tokens -> integer IDs (what the model actually consumes)
token_ids = tokenizer.encode(text)
print(f"IDs       : {token_ids}")
print(f"Length    : {len(token_ids)} tokens")

Input text: "What's a good summer dress under £80?"
Tokens    : ['What', "'s", 'Ġa', 'Ġgood', 'Ġsummer', 'Ġdress', 'Ġunder', 'ĠÂ£', '8', '0', '?']
IDs       : [1780, 506, 253, 1123, 4018, 10272, 656, 10257, 40, 32, 47]
Length    : 11 tokens


Observations:
- Common words → one token each (`What`, `summer`)
- Possessives split (`'s`)
- Punctuation gets its own token
- Currency symbols like `£` may split into a few subword pieces depending on vocab

Every model has its OWN tokenizer. The same input string produces different token sequences for SmolLM2 vs MiniLM vs GPT-2 vs Llama. Don't mix tokenizers across models.

## 3 · The simplest generation — one forward pass

Before we wrap things up in `.generate()`, let's see what ONE forward pass looks like. Feed token IDs, get logits over the vocabulary for the NEXT token.

In [4]:
# Encode and add batch dimension
inputs = tokenizer(text, return_tensors='pt')
print(f"Input IDs shape: {inputs['input_ids'].shape}  (batch=1, seq_len)")

# One forward pass
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits   # shape: (batch, seq_len, vocab_size)
print(f"Logits shape   : {logits.shape}")

# The next-token logits are the LAST position
next_token_logits = logits[0, -1, :]
print(f"Next-token logits shape: {next_token_logits.shape}  (one number per vocab token)")

# Top-5 most likely next tokens
top5 = torch.topk(next_token_logits, 5)
print(f"\nTop 5 candidates for the next token:")
for prob_logit, tok_id in zip(top5.values, top5.indices):
    prob = torch.softmax(next_token_logits, dim=-1)[tok_id]
    print(f"  {tokenizer.decode(tok_id):<20s}  logit={prob_logit:>6.2f}  prob={prob:.3f}")

Input IDs shape: torch.Size([1, 11])  (batch=1, seq_len)
Logits shape   : torch.Size([1, 11, 49152])
Next-token logits shape: torch.Size([49152])  (one number per vocab token)

Top 5 candidates for the next token:
  <|im_end|>            logit= 18.38  prob=0.711
  
                     logit= 16.38  prob=0.096
   I                    logit= 15.88  prob=0.059
   
                    logit= 14.44  prob=0.014
   �                    logit= 14.19  prob=0.011


That's the entire model: 360M parameters processed your input and produced a probability distribution over 49K vocabulary tokens. Pick one, append it, run again. That's text generation.

## 4 · The generation loop, by hand

`model.generate()` does this loop for us. Let's first do it manually to see exactly what's happening.

In [5]:
prompt = "The best way to learn programming is"
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
print(f"Prompt: {prompt!r}")
print(f"Starting with {input_ids.shape[1]} tokens.")

# Greedy generation: pick the argmax token each step
max_new = 20
for step in range(max_new):
    with torch.no_grad():
        out = model(input_ids)
    next_id = out.logits[0, -1, :].argmax().item()
    # append
    input_ids = torch.cat([input_ids, torch.tensor([[next_id]])], dim=1)
    # stop on EOS
    if next_id == tokenizer.eos_token_id:
        break

generated = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(f"\nGenerated:\n  {generated}")

Prompt: 'The best way to learn programming is'
Starting with 7 tokens.

Generated:
  The best way to learn programming is to start with a language that is easy to learn and has a large community of users. Python is


That's the entire generation algorithm. Every modern LLM — GPT-4, Claude, Llama 70B — does fundamentally this. The model is bigger and trained differently, but the loop is identical.

## 5 · Use `model.generate()` for convenience

Doing the loop manually is illustrative but slow. The library's `.generate()` handles batching, caching, and stopping conditions efficiently.

In [6]:
prompt = "The best way to learn programming is"
input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']

t0 = time.time()
output_ids = model.generate(
    input_ids,
    max_new_tokens=40,
    do_sample=False,        # greedy
    pad_token_id=tokenizer.eos_token_id,
)
elapsed = time.time() - t0

generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f"Generated in {elapsed:.2f}s:\n  {generated}")

Generated in 1.57s:
  The best way to learn programming is to start with a language that is easy to learn and has a large community of users. Python is a great language to start with because it is:

Easy to learn: Python has a simple


## 6 · Sampling — temperature, top-k, top-p

Greedy is deterministic and produces boring text. Real applications use sampling. Let's vary the parameters and watch the output change for the same prompt.

In [7]:
# Instruction-tuned models expect chat-formatted prompts. Raw text often
# makes them emit EOS immediately and produce empty output — see Section 7
# of this notebook for the wrong-vs-right comparison. Use apply_chat_template.
messages = [{'role': 'user', 'content': 'Tell me a short bedtime story about a brave little robot.'}]
chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(chat_prompt, return_tensors='pt')['input_ids']

configs = [
    ('Greedy',              dict(do_sample=False)),
    ('Temp=0.7, top-p=0.9', dict(do_sample=True, temperature=0.7, top_p=0.9)),
    ('Temp=1.5, top-p=0.95',dict(do_sample=True, temperature=1.5, top_p=0.95)),
]

for name, cfg in configs:
    torch.manual_seed(42)  # consistent randomness across configs
    out = model.generate(
        input_ids, max_new_tokens=60, pad_token_id=tokenizer.eos_token_id, **cfg
    )
    text = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n=== {name} ===")
    print(text.strip()[:300])


=== Greedy ===
Once upon a time, in a faraway land, there was a tiny robot named Robby. Robby was a brave little robot who loved to explore the world. He had a special suit that made him invisible, so he could sneak around and do all sorts of exciting things.

Robby

=== Temp=0.7, top-p=0.9 ===
Once upon a time, in a place called Robotville, there lived a tiny robot named Robby. Robby was a brave little robot who loved to play with his friends. One day, he decided to go on a mission to save the city from a very big and scary robot named Robo

=== Temp=1.5, top-p=0.95 ===
Livabo lived far away from here - it is up near North Carolina! Livabo didn't need to worry too much anymore though, for in these days lived a big and busy robot that was more than ten times smaller. One sunny afternoon and when Livabo came to his home and found himself


**Notice how temperature controls creativity vs coherence.** Greedy is safest but repetitive. T=1.5 starts to drift but is more interesting. Production sweet spot: T=0.7-0.8 + top-p=0.9 for general use; T=0.2 for code; T=1.0 for creative writing.

Be aware: a 360M model has limits. Don't read too much into the *content* — focus on how each parameter shifts the distribution.

## 7 · Chat templates — what instruction-tuned models actually expect

`SmolLM2-360M-Instruct` was fine-tuned on conversational data. It expects a SPECIFIC format with special tokens marking user/assistant turns. Get the format wrong and you'll get garbage.

In [8]:
# Wrong way: just send the raw question
wrong = "What is a CNN?"
input_ids = tokenizer(wrong, return_tensors='pt')['input_ids']
out = model.generate(input_ids, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print('=== Without chat template (WRONG) ===')
print(tokenizer.decode(out[0], skip_special_tokens=True))

=== Without chat template (WRONG) ===
What is a CNN?


In [9]:
# Right way: apply the chat template
messages = [
    {"role": "user", "content": "What is a CNN?"}
]

# This adds the special <|im_start|> / <|im_end|> tokens and ends with assistant prompt
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print('Formatted prompt:')
print(repr(formatted_prompt))

input_ids = tokenizer(formatted_prompt, return_tensors='pt')['input_ids']
out = model.generate(input_ids, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)
print('\n=== With chat template (RIGHT) ===')
print(response)

Formatted prompt:
'<|im_start|>system\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>\n<|im_start|>user\nWhat is a CNN?<|im_end|>\n<|im_start|>assistant\n'

=== With chat template (RIGHT) ===
A Convolutional Neural Network (CNN) is a type of artificial neural network that is particularly well-suited for image and video processing tasks. It's a type of neural network that uses convolutional layers, pooling layers, and fully connected layers to extract features from input data, such as images or videos.

Convolutional layers are designed to process data at the level of image or video features


See the difference. The chat-templated version produces a clean, focused answer. The raw version drifts.

**Always use `apply_chat_template`** for instruction-tuned models. Don't hand-roll prompts. The model's training defined exactly which special tokens delimit each turn — get it wrong and the model's behaviour is undefined.

## 8 · Multi-turn conversation

In [10]:
# A multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful retail shopping assistant. Answer concisely."},
    {"role": "user", "content": "What's a good fabric for a summer dress?"},
]

def chat(messages, max_new_tokens=120):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids']
    out = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

response = chat(conversation)
print("Assistant:", response)

# Add it to the history and continue
conversation.append({"role": "assistant", "content": response})
conversation.append({"role": "user", "content": "Is linen better than cotton for hot weather?"})

response = chat(conversation)
print("\nAssistant:", response)

Assistant: A good fabric for a summer dress is cotton or linen.

Assistant: Yes, linen is better for hot weather.


**Notice three things:**

1. The **system prompt** sets behaviour ("be a retail assistant, be concise"). You can shape style and constraints here.
2. We **append** the assistant's reply to the conversation history before the next user turn. That's how the model "remembers" what it said.
3. The model gives reasonable, fluent retail-style answers — for a 360M-parameter model, that's remarkable. Imagine what 70B can do.

This is the loop that powers every chatbot, from a customer-service bot to ChatGPT. The model is bigger; the loop is the same.

## 9 · Recap

You now know:

- Tokenisation: text → subword IDs (model-specific)
- Forward pass: IDs → logits → next-token probability
- Generation: iteratively sample/argmax + append until done
- Sampling: temperature, top-k, top-p — tune the creativity dial
- Chat templates: format prompts the way the model expects
- Multi-turn: append history; LLMs are stateless, you supply the state

**Next:** an LLM by itself doesn't know YOUR data. To make it useful for product-specific Q&A, we need to give it the right context. That's RAG. NB 04.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · Streaming output

For interactive applications you want tokens to appear AS the model generates them — not in one big chunk at the end. `TextStreamer` handles this.

In [11]:
from transformers import TextStreamer

prompt = "Write a haiku about cosy autumn knitwear:"
messages = [{"role": "user", "content": prompt}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(formatted, return_tensors='pt')['input_ids']

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
_ = model.generate(input_ids, max_new_tokens=60, do_sample=True, temperature=0.8,
                   top_p=0.9, pad_token_id=tokenizer.eos_token_id, streamer=streamer)
print('\n[end of generation]')

Soft woolen sweater, snugly warm,
In the cozy winter season, snug and cool,
A perfect winter's gift, the fleece.

[end of generation]


## E2 · Stop sequences

You can tell `.generate()` to stop at a specific token sequence. Useful when you want to constrain output format (e.g., 'stop at a closing bracket').

In [12]:
# Stop generation when the model emits the string "STOP_HERE"
prompt_text = "List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.\n1."
input_ids = tokenizer(prompt_text, return_tensors='pt')['input_ids']

# Find the token IDs for the stop string (might be multiple tokens)
stop_ids = tokenizer.encode("STOP_HERE", add_special_tokens=False)
print(f"Stop token IDs: {stop_ids}")

# A simple per-step custom stop function via manual loop
out = input_ids.clone()
for _ in range(80):
    with torch.no_grad():
        nxt = model(out).logits[0, -1, :].argmax().item()
    out = torch.cat([out, torch.tensor([[nxt]])], dim=1)
    # Check if the last few tokens match the stop sequence
    if out[0][-len(stop_ids):].tolist() == stop_ids:
        break

print('Generated:')
print(tokenizer.decode(out[0], skip_special_tokens=True))

Stop token IDs: [48124, 79, 56, 14298]
Generated:
List three Mediterranean salad ingredients, one per line. After the third, write STOP_HERE.
1. *Feta cheese*
2. *Tomato salad*
3. *Balsamic vinaigrette*
4. *Cucumber*
5. *Avocado*
6. *Parmesan cheese*
7. *Bread*
8. *Balsamic glaze*
9. *Ginger*
10. *Cel


Production frameworks (vLLM, llama.cpp) make stop sequences first-class. The mechanic is the same: watch the running output, break when you see what you wanted.